# 🔑 Lấy Drive token cho máy chủ GitHub (chỉ làm 1 lần)
Máy chủ GitHub cần quyền đọc/ghi thư mục **DanhNhan** trên Google Drive của bạn.

1. Secret `YT_CLIENT_JSON` (đã có từ trước) phải bật **Notebook access** cho notebook này (🔑 bên trái).
2. Bấm ▶ chạy ô dưới, làm theo hướng dẫn hiện ra.
3. Chép **refresh token** vào GitHub Secret tên `GDRIVE_REFRESH_TOKEN`.


In [ ]:
# Cấp quyền Google Drive cho máy chủ — chạy 1 lần
import json
from urllib.parse import urlparse, parse_qs
from google.colab import userdata, output
from google_auth_oauthlib.flow import Flow

SCOPE = 'https://www.googleapis.com/auth/drive'
try:
    client = json.loads(userdata.get('YT_CLIENT_JSON'))
except Exception:
    raise SystemExit('❌ Không đọc được Secret YT_CLIENT_JSON — bật Notebook access cho notebook này (🔑 bên trái).')
flow = Flow.from_client_config(client, scopes=[SCOPE], redirect_uri='http://localhost')
url, _ = flow.authorization_url(access_type='offline', prompt='consent', include_granted_scopes='false')
print('① Mở link dưới, đăng nhập ĐÚNG tài khoản Google chứa thư mục DanhNhan.')
print('   Gặp màn hình "Google hasn\'t verified this app" → Advanced (Nâng cao) → Go to … (unsafe) → Continue/Cho phép.\n')
print(url, '\n')
print('② Trình duyệt chuyển sang trang lỗi "localhost" (bình thường). Chép TOÀN BỘ địa chỉ (http://localhost/?state=…) dán xuống dưới.\n')
while True:
    resp = input('Dán địa chỉ vào đây rồi Enter: ').strip()
    q = parse_qs(urlparse(resp).query)
    if resp.startswith(('http://localhost', 'localhost')) and 'code' in q:
        break
    if 'error' in q:
        raise SystemExit(f"❌ Google báo: {q['error'][0]} — chạy lại ô này và bấm Cho phép.")
    print('⚠ Chưa đúng: địa chỉ phải bắt đầu bằng http://localhost/?state= và có code=4/ . Thử lại.\n')
try:
    flow.fetch_token(code=q['code'][0])
except Exception as e:
    raise SystemExit(f'❌ Mã cấp quyền hết hạn hoặc đã dùng ({e}). Chạy lại ô này.')
rt = flow.credentials.refresh_token
if not rt:
    raise SystemExit('❌ Google không trả về refresh token — chạy lại ô này.')
print('\n③ GDRIVE_REFRESH_TOKEN (bí mật như mật khẩu, KHÔNG gửi cho ai):\n')
print(rt)
print('\n④ GitHub → kho biography-factory → Settings → Secrets and variables → Actions → New repository secret')
print('   Name: GDRIVE_REFRESH_TOKEN   Secret: dán token trên')
input('\nChép xong thì bấm Enter để XOÁ token khỏi màn hình… ')
output.clear(); print('✅ Đã xoá token khỏi màn hình.')
